In [1]:
# selenium, Beautifulsoup 임포트
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as ec
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support.ui import Select
from bs4 import BeautifulSoup
import time

# 대기 시간 변수 선언
sleep_time = 2

In [ ]:
# 데이터별 리스트 초기화(시/도 구분없이 한 리스트에 전부 저장)
name_lst = [] # 이름 리스트
address_lst = [] # 도로명 주소 리스트
g_price_lst = [] # 휘발유 가격 리스트
d_price_lst = [] # 경유 가격 리스트
phone_lst = [] # 전화번호 리스트

# selenium 드라이버 객체 생성 및 페이지 불러오기
driver = webdriver.Chrome()
driver.get('https://www.opinet.co.kr/searRgOsSelect.do')
WebDriverWait(driver, sleep_time).until(ec.presence_of_element_located((By.ID, 'SIDO_NM0')))

# 시/도 select_box의 전체 옵션 수
option_cnt = len(Select(driver.find_element(By.ID, 'SIDO_NM0')).options)

# 전국 시/도 착한주유소 이름, 휘발유 가격, 경유 가격, 주소 크롤링 루프문
for i in range(1, option_cnt):
    # 시/도 선택하는 select_box 객체 생성
    select_element = driver.find_element(By.ID, 'SIDO_NM0')
    select_box = Select(select_element)

    # 시/도 선택
    select_box.select_by_index(i)

    # 조회 버튼 클릭
    driver.find_element(By.ID, 'searRgSelect').click()
    time.sleep(sleep_time) # 페이지 요소들 로딩될때까지 대기
    
    # 시의 착한 주유소 세부정보 크롤링
    # 주유소 세부정보 링크 객체 리스트
    WebDriverWait(driver, sleep_time).until(ec.presence_of_element_located((By.CSS_SELECTOR, '.rlist > a')))
    href_elements = driver.find_elements(By.CSS_SELECTOR, '.rlist > a')

    # 각 주유소 세부정보 링크 클릭 -> 세부정보 크롤링 루프문
    for i in range(len(href_elements)):
        try:
            # 링크 클릭
            href_elements[i].click()

            # Beautiful soup 객체 생성
            results_html = driver.page_source
            soup = BeautifulSoup(results_html, 'html.parser')

            # 주소 문자열 추출 및 리스트 저장
            WebDriverWait(driver, sleep_time).until(ec.presence_of_element_located((By.ID, 'rd_addr')))
            address = soup.select_one('#rd_addr').text
            address_lst.append(address)

            # 이름 문자열 추출 및 리스트 저장
            WebDriverWait(driver, sleep_time).until(ec.presence_of_element_located((By.ID, 'os_nm')))
            name = soup.select_one('#os_nm').text
            name_lst.append(name)

            # 휘발유 가격 추출 및 리스트 저장
            WebDriverWait(driver, sleep_time).until(ec.presence_of_element_located((By.ID, 'b027_p')))
            g_price = soup.select_one('#b027_p').text
            g_price_lst.append(g_price)
            
            # 경유 가격 추출 및 리스트 저장
            WebDriverWait(driver, sleep_time).until(ec.presence_of_element_located((By.ID, 'd047_p')))
            d_price = soup.select_one('#d047_p').text
            d_price_lst.append(d_price)

            # 전화번호 추출 및 리스트 저장
            WebDriverWait(driver, sleep_time).until(ec.presence_of_element_located((By.ID, 'phn_no')))
            phone = soup.select_one('#phn_no').text
            phone_lst.append(phone)
        
        except:
            continue
        
# print(len(name_lst))
# print(len(g_price_lst))
# print(len(d_price_lst))
# print(len(address_lst))
# print(len(phone_lst))
# print(name_lst)
# print(g_price_lst)
# print(d_price_lst)
# print(address_lst)
# print(phone_lst)

# 시/도별로 저장하고 싶으면 루프마다 name_lst, g_price_lst, d_price_lst, phone_lst 초기화 후 db에 저장


